# Retrieval-Augmented Generation (RAG): Полностью рабочий ноутбук с LangGraph

В этом ноутбуке вы сможете:
- Поднять и инициализировать базу данных (PostgreSQL)
- Загрузить статьи ТК РФ
- Построить и использовать векторный индекс (FAISS/pgvector)
- Реализовать разные RAG-подходы: от простого LLM до мультиагентных систем
- Использовать LangGraph для построения сложных пайплайнов

> **Важно:** Все шаги воспроизводимы и могут быть выполнены прямо из ноутбука. Следуйте инструкциям и запускайте ячейки по порядку.

## 1. Базовый LLM-ответ без Retrieval

**Описание:**
Модель LLM отвечает на вопрос пользователя, не используя внешний контекст или документы. Такой подход прост, но ограничен знаниями самой модели.

**Плюсы:**
- Быстро
- Не требует подготовки данных

**Минусы:**
- Нет актуальности
- Модель может "выдумывать" ответы

**Когда использовать:**
- Для общих вопросов, когда точность не критична
- В тестовых прототипах

**Пример кода:**

## 0. Подготовка окружения и данных

Перед началом работы убедитесь, что у вас установлен Docker и все зависимости проекта. Далее выполните подготовку базы данных и загрузку статей прямо из ноутбука.

In [11]:
# Запуск PostgreSQL и подготовка данных (можно запускать прямо из ноутбука)
!make db-up
!make db-init
!make db-load

8964.56s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


make: *** No rule to make target 'db-up'.  Stop.


8969.80s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


make: *** No rule to make target 'db-init'.  Stop.


8975.02s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


make: *** No rule to make target 'db-load'.  Stop.


---

## 1. Базовый LLM-ответ без Retrieval

**Описание:**
Модель LLM отвечает на вопрос пользователя, не используя внешний контекст или документы. Такой подход прост, но ограничен знаниями самой модели.

**Плюсы:**
- Быстро
- Не требует подготовки данных

**Минусы:**
- Нет актуальности
- Модель может "выдумывать" ответы

**Когда использовать:**
- Для общих вопросов, когда точность не критична
- В тестовых прототипах

**Пример кода:**

In [14]:
import os
import sys

sys.path.append(os.path.abspath(".."))

# Установите прокси до импорта LLMService
os.environ["ALL_PROXY"] = "socks5://127.0.0.1:12334"
os.environ["all_proxy"] = "socks5://127.0.0.1:12334"

from src.application.services.llm_service import LLMService

api_key = os.getenv("GROQ_API_KEY")
llm = LLMService(api_key=api_key, proxy="socks5://127.0.0.1:12334")
question = "Может ли работодатель уволить сотрудника за прогул?"
answer = await llm.generate_answer(question, articles=[])
print(answer.answer)

INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Да, работодатель может уволить сотрудника за прогул. Согласно статье 81 ТК РФ, увольнение по инициативе работодателя допускается в случаях, предусмотренных настоящим Кодексом, в том числе за неоднократное неисполнение сотрудником без уважительных причин своих трудовых обязанностей (прогул).

Статья 192 ТК РФ также гласит, что прогул является нарушением трудовой дисциплины и может быть основанием для применения дисциплинарного взыскания, вплоть до увольнения.

Таким образом, прогул может быть основанием для увольнения сотрудника, но работодатель должен соблюдать все необходимые процедуры и требования, установленные Трудовым кодексом РФ.


In [15]:
# --- Пример настройки прокси для LLM (Groq/LangChain) ---
import os

proxy = "socks5://127.0.0.1:12334"  # Укажите свой прокси, если требуется

# Сохраняем оригинальные значения
original_all_proxy = os.environ.get("ALL_PROXY")
original_all_proxy_lower = os.environ.get("all_proxy")

# Устанавливаем прокси для всех переменных окружения
os.environ["ALL_PROXY"] = proxy
os.environ["all_proxy"] = proxy

# Теперь можно импортировать и использовать LLMService или LangChain
# ... (основной код работы с LLM)

# После работы восстановите переменные окружения
if original_all_proxy is not None:
    os.environ["ALL_PROXY"] = original_all_proxy
if original_all_proxy_lower is not None:
    os.environ["all_proxy"] = original_all_proxy_lower

---

## 2. RAG с простым семантическим поиском (Vector Search)

**Описание:**
Используется векторный поиск для нахождения релевантных документов (например, статей ТК РФ). Найденные статьи добавляются в контекст для LLM, что повышает точность и актуальность ответа.

**Плюсы:**
- Более точные ответы
- Использование актуальных данных

**Минусы:**
- Требуется подготовка индекса
- Качество зависит от поиска

**Когда использовать:**
- Для задач, где важна актуальность и точность
- При наличии базы документов

**Пример кода:**

In [20]:
import os
import sys

sys.path.append(os.path.abspath(".."))

# Установите прокси до импорта LLMService и VectorService
os.environ["ALL_PROXY"] = "socks5://127.0.0.1:12334"
os.environ["all_proxy"] = "socks5://127.0.0.1:12334"

from src.application.services.vector_service import VectorService, VectorBackend
from src.infrastructure.repositories import ArticleRepository
from src.application.services.llm_service import LLMService
from src.infrastructure.database.session import get_db_session

# Получаем API-ключ
api_key = os.getenv("GROQ_API_KEY")
if not api_key:
    print("x Отсутствует ключ api_key")


async def main():
    async with get_db_session() as session:
        repo = ArticleRepository(session)
        articles = await repo.get_all(limit=1000)
        print(f"Загружено статей: {len(articles)}")
        if not articles:
            print(
                "❌ Нет статей в базе. Проверьте выполнение make db-load и содержимое таблицы."
            )
            return

        # 1. Построение индекса статей
        vector_service = VectorService(backend=VectorBackend.FAISS)
        await vector_service.build_index(articles)

        # 2. Поиск релевантных статей
        query = "Увольнение за прогул"
        relevant_articles = await vector_service.find_similar(query, top_k=3)

        # 3. Генерация ответа с контекстом
        llm = LLMService(api_key=api_key, proxy="socks5://127.0.0.1:12334")
        answer = await llm.generate_answer(query, articles=relevant_articles)
        print(answer.answer)


await main()

INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: cpu
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


Загружено статей: 537


Batches: 100%|██████████| 17/17 [00:09<00:00,  1.77it/s]


✓ Built VectorBackend.FAISS index with 537 articles, dimension=384


Batches: 100%|██████████| 1/1 [00:00<00:00, 128.18it/s]
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


К сожалению, предоставленные статьи ТК РФ не содержат информации об увольнении за прогул. Статьи 77, 80 и 280 ТК РФ описывают общие основания прекращения трудового договора, расторжение трудового договора по инициативе работника и досрочное расторжение трудового договора по инициативе руководителя организации соответственно, но не упоминают прогул как основание для увольнения.

Для получения информации об увольнении за прогул необходимо обратиться к другим статьям ТК РФ, в частности к статье 81, которая описывает основания для расторжения трудового договора по инициативе работодателя, включая неоднократное неисполнение работником без уважительных причин своих трудовых обязанностей.


---

## 3. RAG с расширенным контекстом (Advanced RAG)

**Описание:**
После поиска документов проводится дополнительная фильтрация по порогу релевантности, объединение и обработка контекста, возможно, reranking. Это позволяет повысить качество ответа.

**Плюсы:**
- Максимальная релевантность
- Гибкость настройки

**Минусы:**
- Сложнее реализовать
- Требует экспериментов с порогами и стратегиями

**Когда использовать:**
- Для сложных задач, где важна точность
- В production-системах

**Пример кода:**

In [27]:
import os
import sys

sys.path.append(os.path.abspath(".."))

# Установите прокси до импорта LLMService и VectorService
os.environ["ALL_PROXY"] = "socks5://127.0.0.1:12334"
os.environ["all_proxy"] = "socks5://127.0.0.1:12334"

from src.infrastructure import get_db_session, ArticleRepository
from src.application.services.llm_service import LLMService

api_key = os.getenv("GROQ_API_KEY")
if not api_key:
    print("x Отсутствует ключ api_key")


async def main():
    async with get_db_session() as session:
        repo = ArticleRepository(session)
        articles = await repo.get_all(limit=1000)
        print(f"Загружено статей: {len(articles)}")
        if not articles:
            print(
                "❌ Нет статей в базе. Проверьте выполнение make db-load и содержимое таблицы."
            )
            return

        # 1. Построение индекса статей
        vector_service = VectorService(backend=VectorBackend.FAISS)
        await vector_service.build_index(articles)

        # 2. Поиск и фильтрация релевантных статей
        query = "Порядок увольнения за прогул"
        distances, indices = await vector_service.store.search(
            vector_service._load_model().encode([query]), top_k=10
        )
        print(f"Дистанции: {distances}")
        print(f"Индексы: {indices}")
        # Порог подбирается экспериментально и зависит от модели/метрики
        threshold = 10.0
        filtered_articles = [
            articles[idx] for idx, dist in zip(indices, distances) if dist < threshold
        ]

        print(f"Найдено релевантных статей: {len(filtered_articles)}")
        if not filtered_articles:
            print(
                "❌ Не найдено релевантных статей. Попробуйте изменить запрос или порог."
            )
            # Для отладки: покажем хотя бы одну статью
            if len(indices) > 0:
                print("Первые 3 статьи:")
                for idx in indices[:3]:
                    print(f"Статья: {articles[idx].title[:80]}")
            return

        # 3. Генерация ответа с расширенным контекстом
        llm = LLMService(api_key=api_key, proxy="socks5://127.0.0.1:12334")
        answer = await llm.generate_answer(query, articles=filtered_articles)
        print(answer.answer)


await main()

INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: cpu
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


Загружено статей: 537


Batches: 100%|██████████| 17/17 [00:09<00:00,  1.76it/s]


✓ Built VectorBackend.FAISS index with 537 articles, dimension=384


Batches: 100%|██████████| 1/1 [00:00<00:00, 87.20it/s]


Дистанции: [7.892332077026367, 8.089790344238281, 8.343914031982422, 8.800095558166504, 8.918285369873047, 8.930255889892578, 9.081780433654785, 9.163183212280273, 9.319509506225586, 9.374307632446289]
Индексы: [325, 92, 95, 100, 94, 96, 155, 99, 87, 508]
Найдено релевантных статей: 10


INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Отвечаю на ваш вопрос на основе статьи 81 ТК РФ.

Прогул является однократным грубым нарушением трудовых обязанностей, которое может стать основанием для увольнения работника по инициативе работодателя (п. 6 ч. 1 ст. 81 ТК РФ). Прогул определяется как отсутствие на рабочем месте без уважительных причин в течение всего рабочего дня (смены) или более четырех часов подряд в течение рабочего дня (смены).

Для увольнения работника по этому основанию работодатель должен предупредить его о возможности увольнения, а также провести процедуру увольнения в соответствии со статьей 84.1 ТК РФ.

Ссылка: Статья 81 ТК РФ.


---

## 4. RAG с агентным оркестратором (Agent-Orchestrated RAG)

**Описание:**
Используется агент (например, ConversationOrchestrator), который управляет историей диалога, выбирает стратегию поиска, агрегирует источники и формирует финальный ответ.

**Плюсы:**
- Гибкая логика обработки
- Возможность интеграции нескольких стратегий

**Минусы:**
- Требует сложной архитектуры
- Сложнее тестировать

**Когда использовать:**
- Для сложных диалоговых систем
- Когда нужно комбинировать несколько подходов

**Пример кода:**

In [29]:
# Пример: Agent-Orchestrated RAG (рабочий, с правильной инициализацией)
import os
import sys

sys.path.append(os.path.abspath(".."))

# Установите прокси до импорта сервисов
os.environ["ALL_PROXY"] = "socks5://127.0.0.1:12334"
os.environ["all_proxy"] = "socks5://127.0.0.1:12334"

from src.infrastructure import ConversationRepository
from src.application.services.llm_service import LLMService
from src.application.use_cases import (
    ConversationOrchestratorUseCase,
    ManageConversationUseCase,
    AnswerLegalQuestionUseCase,
    MultiStrategySearchUseCase,
)
from src.domain.entities import LegalQuery

api_key = os.getenv("GROQ_API_KEY")
if not api_key:
    print("❌ Отсутствует ключ api_key")


async def main():
    async with get_db_session() as session:
        article_repo = ArticleRepository(session)
        conversation_repo = ConversationRepository(session)
        articles = await article_repo.get_all(limit=1000)
        print(f"Загружено статей: {len(articles)}")
        if not articles:
            print(
                "❌ Нет статей в базе. Проверьте выполнение make db-load и содержимое таблицы."
            )
            return

        # Сервисы
        vector_service = VectorService(backend=VectorBackend.FAISS)
        await vector_service.build_index(articles)
        llm_service = LLMService(api_key=api_key, proxy="socks5://127.0.0.1:12334")

        # Use cases
        manage_conversation_uc = ManageConversationUseCase(conversation_repo)
        multi_strategy_search_uc = MultiStrategySearchUseCase(
            article_repo, vector_service
        )
        answer_legal_question_uc = AnswerLegalQuestionUseCase(
            multi_strategy_search=multi_strategy_search_uc,
            llm_service=llm_service,
            conversation_repository=conversation_repo,
        )
        orchestrator = ConversationOrchestratorUseCase(
            answer_use_case=answer_legal_question_uc,
            manage_conversation_use_case=manage_conversation_uc,
        )
        query = LegalQuery(question="Права работника при увольнении", user_id="123")
        answer = await orchestrator.handle_question(query)
        print(answer.answer)
        print("Источники:", answer.article_numbers)


await main()

INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: cpu
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


Загружено статей: 537


Batches: 100%|██████████| 17/17 [00:09<00:00,  1.76it/s]


✓ Built VectorBackend.FAISS index with 537 articles, dimension=384


Batches: 100%|██████████| 1/1 [00:00<00:00, 111.05it/s]
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


При увольнении работник имеет следующие права:

1. **Получение выходного пособия**: в случае увольнения в связи с ликвидацией организации или сокращением численности или штата работников организации, работник имеет право на получение выходного пособия в размере среднего месячного заработка (Статья 318 ТК РФ).
2. **Получение среднего месячного заработка за период трудоустройства**: в случае увольнения в связи с ликвидацией организации или сокращением численности или штата работников организации, работник имеет право на получение среднего месячного заработка за период трудоустройства (Статья 318 ТК РФ).
3. **Получение единовременной компенсации**: работодатель может выплатить работнику единовременную компенсацию в размере пятикратного среднего месячного заработка вместо выплат среднего месячного заработка за период трудоустройства (Статья 318 ТК РФ).
4. **Получение сведений о трудовой деятельности**: работодатель обязан выдать работнику сведения о трудовой деятельности (Статья 66.1 ТК РФ

---

## 5. Мультиагентная RAG-система (Multi-Agent RAG)

**Описание:**
Используется несколько агентов, каждый из которых отвечает за свою задачу: поиск документов, генерация ответа, проверка достоверности, агрегирование результатов.

**Плюсы:**
- Высокая гибкость и масштабируемость
- Возможность параллельной работы

**Минусы:**
- Сложная реализация и координация
- Требует продуманной архитектуры

**Когда использовать:**
- Для сложных систем с несколькими этапами обработки
- В задачах, где важна проверка и агрегирование информации

**Пример кода:**

In [30]:
# Пример: Multi-Agent RAG (практический, рабочий)
import os
import sys

sys.path.append(os.path.abspath(".."))

# Установите прокси до импорта сервисов
os.environ["ALL_PROXY"] = "socks5://127.0.0.1:12334"
os.environ["all_proxy"] = "socks5://127.0.0.1:12334"

from src.application.services.llm_service import LLMService

api_key = os.getenv("GROQ_API_KEY")
if not api_key:
    print("❌ Отсутствует ключ api_key")


async def main():
    async with get_db_session() as session:
        article_repo = ArticleRepository(session)
        articles = await article_repo.get_all(limit=1000)
        print(f"Загружено статей: {len(articles)}")
        if not articles:
            print(
                "❌ Нет статей в базе. Проверьте выполнение make db-load и содержимое таблицы."
            )
            return

        # Агент поиска (векторный)
        vector_agent = VectorService(backend=VectorBackend.FAISS)
        await vector_agent.build_index(articles)

        # Агент генерации ответа
        llm_agent = LLMService(api_key=api_key, proxy="socks5://127.0.0.1:12334")

        # Агент проверки достоверности
        def verify_answer(answer):
            # Проверка на наличие источников, соответствие политике
            return "статья" in answer.answer or answer.article_numbers

        # Координация агентов
        query = "Права работника при увольнении"
        relevant_articles = await vector_agent.find_similar(query, top_k=5)
        answer = await llm_agent.generate_answer(query, articles=relevant_articles)
        if verify_answer(answer):
            print("Ответ достоверен:\n", answer.answer)
            print("Источники:", answer.article_numbers)
        else:
            print("Ответ требует проверки или доработки.")


await main()

INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: cpu
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


Загружено статей: 537


Batches: 100%|██████████| 17/17 [00:09<00:00,  1.70it/s]


✓ Built VectorBackend.FAISS index with 537 articles, dimension=384


Batches: 100%|██████████| 1/1 [00:00<00:00, 127.63it/s]
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Ответ достоверен:
 При увольнении работник имеет определенные права, которые регулируются Трудовым кодексом РФ. 

Согласно статье 80 ТК РФ, работник имеет право расторгнуть трудовой договор, предупредив об этом работодателя в письменной форме не позднее чем за две недели. 

Также, согласно статье 318 ТК РФ, работникам, увольняемым в связи с ликвидацией организации либо сокращением численности или штата работников организации, выплачивается выходное пособие в размере среднего месячного заработка.

Кроме того, работодатель обязан выдать работнику трудовую книжку или предоставить сведения о трудовой деятельности, выдать другие документы, связанные с работой, и произвести с ним окончательный расчет (статья 80 ТК РФ).

При увольнении работник также имеет право на получение компенсационных выплат, если они предусмотрены трудовым договором или коллективным договором (статья 307 ТК РФ).

Следует отметить, что права работника при увольнении могут варьироваться в зависимости от конкретных обстоя

---

## 6. Multi-Agent RAG с LangGraph

**Описание:**
LangGraph позволяет строить сложные графы агентов и инструментов для RAG-системы. Можно явно задавать последовательность и ветвления между поиском, rerank, генерацией, верификацией, внешними инструментами (web search, калькулятор и др.).

**Преимущества:**
- Гибкая композиция агентов и инструментов
- Явное описание логики пайплайна (графа)
- Легко добавлять новые ветки, fallback, внешние источники

**Пример кода:**


In [45]:
import os
import sys
import traceback
from typing import TypedDict, List, Optional, Literal

# Импорт LangGraph
from langgraph.graph import StateGraph, END

# ВАЖНО для Notebook: позволяем вложенный asyncio
# !pip install nest_asyncio
import nest_asyncio

nest_asyncio.apply()


# --- Схема состояния ---
class RagState(TypedDict):
    """
    Используем TypedDict — это критически важно для LangGraph в новых версиях,
    чтобы он корректно инициализировал ключи состояния.
    """

    query: str
    articles: List[any]
    answer: Optional[any]
    retry_count: int


# --- Узлы и логика ---


def create_rag_graph(article_repo, vector_service, llm_service):
    # 1. Поиск
    async def search_node(state: RagState):
        print(f"DEBUG: [search_node] Ищем статьи для: '{state.get('query')}'")
        articles = await article_repo.get_all(limit=1000)
        await vector_service.build_index(articles)
        found = await vector_service.find_similar(state["query"], top_k=5)
        return {"articles": found}

    # 2. Генерация
    async def generate_node(state: RagState):
        print(
            f"DEBUG: [generate_node] Генерируем ответ по {len(state.get('articles', []))} статьям"
        )
        # Здесь мы вызываем ваш llm_service
        ans = await llm_service.generate_answer(
            state["query"], articles=state["articles"]
        )
        current_retries = state.get("retry_count", 0)
        return {"answer": ans, "retry_count": current_retries + 1}

    # 3. Роутер (Проверка)
    def should_continue(state: RagState) -> Literal["search", "__end__"]:
        ans = state.get("answer")
        # Ваша логика верификации
        is_valid = bool(
            ans and (hasattr(ans, "answer") and "статья" in ans.answer.lower())
        )

        if is_valid:
            print("DEBUG: [router] Ответ валиден. Завершаем.")
            return END

        if state.get("retry_count", 0) >= 2:
            print("DEBUG: [router] Попытки исчерпаны. Выходим с тем, что есть.")
            return END

        print("DEBUG: [router] Ответ не прошел проверку. Повторный поиск...")
        return "search"

    # --- Сборка графа ---
    workflow = StateGraph(RagState)

    workflow.add_node("search", search_node)
    workflow.add_node("generate", generate_node)

    workflow.set_entry_point("search")
    workflow.add_edge("search", "generate")

    workflow.add_conditional_edges(
        "generate", should_continue, {"search": "search", END: END}
    )

    return workflow.compile()


# --- Запуск ---


async def run_workflow():
    # Настройки
    api_key = os.getenv("GROQ_API_KEY")
    if not api_key:
        print("Ошибка: Установите GROQ_API_KEY")
        return

    # Инициализация ваших сервисов внутри сессии
    async with get_db_session() as session:
        article_repo = ArticleRepository(session)
        vector_service = VectorService(backend=VectorBackend.FAISS)
        llm_service = LLMService(api_key=api_key, proxy="socks5://127.0.0.1:12334")

        # Создаем граф
        app = create_rag_graph(article_repo, vector_service, llm_service)

        # Начальный вход
        inputs = {
            "query": "Какие права у работника при сокращении?",
            "articles": [],
            "answer": None,
            "retry_count": 0,
        }

        print("--- ЗАПУСК ГРАФА ---")
        try:
            # В ноутбуке используем await напрямую
            final_state = await app.ainvoke(inputs)

            print("\n" + "=" * 50)
            if final_state.get("answer"):
                print("ИТОГОВЫЙ ОТВЕТ:")
                print(final_state["answer"].answer)
            else:
                print("Ответ не удалось получить.")
            print("=" * 50)

        except Exception:
            traceback.print_exc()


# В Юпитере вызываем через await
await run_workflow()

INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: cpu
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


--- ЗАПУСК ГРАФА ---
DEBUG: [search_node] Ищем статьи для: 'Какие права у работника при сокращении?'


Batches: 100%|██████████| 17/17 [00:11<00:00,  1.53it/s]


✓ Built VectorBackend.FAISS index with 537 articles, dimension=384


Batches: 100%|██████████| 1/1 [00:00<00:00, 56.40it/s]


DEBUG: [generate_node] Генерируем ответ по 5 статьям


INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


DEBUG: [router] Ответ валиден. Завершаем.

ИТОГОВЫЙ ОТВЕТ:
При сокращении численности или штата работников работник имеет право на преимущественное оставление на работе, если он имеет более высокую производительность труда и квалификацию (Статья 179 ТК РФ). 

Если производительность труда и квалификация равны, предпочтение отдается определенным категориям работников, таким как:

- Семейные, имеющие двух или более иждивенцев
- Лица, в семье которых нет других работников с самостоятельным заработком
- Работники, получившие в период работы у данного работодателя трудовое увечье или профессиональное заболевание
- Инвалиды Великой Отечественной войны и инвалиды боевых действий по защите Отечества
- Работники, повышающие свою квалификацию по направлению работодателя без отрыва от работы
- Родитель, имеющий ребенка в возрасте до восемнадцати лет, в случае, если другой родитель призван на военную службу по мобилизации или направлен на службу в войска национальной гвардии Российской Федерации п

### 7. Self‑Corrective RAG (CRAG) — Самокорректирующаяся система

**Основные отличия от базовой версии:**

- **Узел Grader (Оценщик):** не доверяем поиску слепо — отдельный шаг оценивает релевантность найденных документов и решает, достаточно ли их для генерации ответа.  
- **Узел Transform Query:** если поиск вернул нерелевантные результаты, LLM переформулирует запрос в более строгий технический/юридический вид для повторного поиска.  
- **Автономный выбор пути:** граф сам принимает решение — идти на генерацию ответа или вернуться к перепоиску/трансформации запроса (включая защиту от зацикливания через счётчик попыток).

In [46]:
import os
import sys
from typing import TypedDict, List, Optional

# Импорт LangGraph

# ВАЖНО для Notebook: позволяем вложенный asyncio
import nest_asyncio

nest_asyncio.apply()


# 1. Схема состояния (Advanced State)
class CRAGState(TypedDict):
    """
    query: Исходный вопрос пользователя
    transformed_query: Переработанный вопрос для поиска
    articles: Список найденных документов
    answer: Финальный ответ
    loop_step: Счетчик итераций (защита от зацикливания)
    """

    query: str
    transformed_query: Optional[str]
    articles: List[any]
    answer: Optional[any]
    loop_step: int


# --- ОПРЕДЕЛЕНИЕ УЗЛОВ ГРАФА ---


def create_crag_graph(article_repo, vector_service, llm_service):
    # Узел 1: Поиск документов
    async def retrieve_node(state: CRAGState):
        # Используем трансформированный запрос, если он есть, иначе оригинал
        current_query = state.get("transformed_query") or state["query"]
        step = state.get("loop_step", 0)

        print(f"DEBUG: [Step {step}] Поиск статей для: '{current_query}'")

        # Ваша логика поиска
        all_articles = await article_repo.get_all(limit=1000)
        await vector_service.build_index(all_articles)
        found = await vector_service.find_similar(current_query, top_k=5)

        return {"articles": found, "loop_step": step + 1}

    # Узел 2: Оценка релевантности (Grader)
    def grade_documents_logic(
        state: CRAGState,
    ) -> Literal["generate", "transform", "finalize"]:
        articles = state.get("articles", [])

        # Если ничего не нашли или превысили лимит попыток
        if not articles:
            if state["loop_step"] >= 2:
                print("DEBUG: [Grader] Документы не найдены, попытки исчерпаны.")
                return "finalize"
            print("DEBUG: [Grader] Результатов нет. Отправка на трансформацию запроса.")
            return "transform"

        # Логика проверки: если в статьях нет ключевого слова из запроса (упрощенно)
        # В идеале здесь вызывается легкая LLM-модель для оценки (Grader)
        relevance_score = 1  # Допустим, мы нашли что-то полезное

        if relevance_score > 0:
            print(
                f"DEBUG: [Grader] Найдено {len(articles)} документов. Идем на генерацию."
            )
            return "generate"
        else:
            print("DEBUG: [Grader] Документы не релевантны. Трансформируем запрос.")
            return "transform"

    # Узел 3: Трансформация запроса
    async def transform_query_node(state: CRAGState):
        print("DEBUG: [Transform] Переписываем запрос для улучшения поиска...")
        original_query = state["query"]

        # Здесь мы заставляем LLM перефразировать запрос
        # Пример промпта: "Перепиши запрос для юридической базы данных: {original_query}"
        # Для примера просто добавим ключевые слова
        new_query = f"нормативные акты и права: {original_query}"

        return {"transformed_query": new_query}

    # Узел 4: Генерация финального ответа
    async def generate_node(state: CRAGState):
        print("DEBUG: [Generate] Формируем итоговый ответ на основе контекста...")
        ans = await llm_service.generate_answer(
            state["query"], articles=state["articles"]
        )
        return {"answer": ans}

    # Узел 5: Пустой ответ (если ничего не нашли)
    async def finalize_empty_node(state: CRAGState):
        print("DEBUG: [Finalize] Информация не найдена в базе данных.")
        return {
            "answer": "К сожалению, в нашей базе данных нет информации по вашему вопросу."
        }

    # --- СБОРКА ГРАФА ---
    workflow = StateGraph(CRAGState)

    # Добавляем все узлы
    workflow.add_node("retrieve", retrieve_node)
    workflow.add_node("transform_query", transform_query_node)
    workflow.add_node("generate", generate_node)
    workflow.add_node("finalize_empty", finalize_empty_node)

    # Логика переходов
    workflow.set_entry_point("retrieve")

    # Условный переход после поиска (CRAG logic)
    workflow.add_conditional_edges(
        "retrieve",
        grade_documents_logic,
        {
            "generate": "generate",
            "transform": "transform_query",
            "finalize": "finalize_empty",
        },
    )

    # Если трансформировали запрос — возвращаемся на поиск
    workflow.add_edge("transform_query", "retrieve")

    # Конечные точки
    workflow.add_edge("generate", END)
    workflow.add_edge("finalize_empty", END)

    return workflow.compile()


# --- ФУНКЦИЯ ЗАПУСКА В ЯЧЕЙКЕ ---


async def run_crag_example():
    # Настройки API (замени на свои)
    api_key = os.getenv("GROQ_API_KEY")

    # 1. Инициализация вашей инфраструктуры
    async with get_db_session() as session:
        article_repo = ArticleRepository(session)
        vector_service = VectorService(backend=VectorBackend.FAISS)
        llm_service = LLMService(api_key=api_key, proxy="socks5://127.0.0.1:12334")

        # 2. Создаем и компилируем граф
        app = create_crag_graph(article_repo, vector_service, llm_service)

        # 3. Входные данные
        # Попробуем сложный запрос, который может потребовать трансформации
        inputs = {
            "query": "как уйти по собственному без отработки",
            "articles": [],
            "loop_step": 0,
            "transformed_query": None,
        }

        print("--- СТАРТ SELF-CORRECTIVE RAG ---")
        try:
            # Запуск графа
            final_state = await app.ainvoke(inputs)

            print("\n" + "=" * 50)
            print("ИТОГОВЫЙ ОТВЕТ:")
            # Проверяем, есть ли у ответа атрибут answer или это просто строка
            ans_obj = final_state.get("answer")
            if hasattr(ans_obj, "answer"):
                print(ans_obj.answer)
            else:
                print(ans_obj)
            print("=" * 50)

        except Exception:
            print("Произошла ошибка при выполнении графа:")
            traceback.print_exc()


# Запуск в ноутбуке
await run_crag_example()

--- СТАРТ SELF-CORRECTIVE RAG ---
DEBUG: [Step 0] Поиск статей для: 'как уйти по собственному без отработки'


INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: cpu
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Batches: 100%|██████████| 17/17 [00:10<00:00,  1.62it/s]


✓ Built VectorBackend.FAISS index with 537 articles, dimension=384


Batches: 100%|██████████| 1/1 [00:00<00:00, 47.22it/s]


DEBUG: [Grader] Найдено 5 документов. Идем на генерацию.
DEBUG: [Generate] Формируем итоговый ответ на основе контекста...


INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"



ИТОГОВЫЙ ОТВЕТ:
Согласно статье 80 ТК РФ, работник имеет право расторгнуть трудовой договор, предупредив об этом работодателя в письменной форме не позднее чем за две недели. Однако, в некоторых случаях работодатель обязан расторгнуть трудовой договор в срок, указанный в заявлении работника, без отработки двух недель. 

Такие случаи включают:

- невозможность продолжения работы (зачисление в образовательную организацию, выход на пенсию и другие случаи)
- нарушение работодателем трудового законодательства и иных нормативных правовых актов, содержащих нормы трудового права, локальных нормативных актов, условий коллективного договора, соглашения или трудового договора.

Если ваша ситуация подпадает под один из этих случаев, вы можете обратиться к работодателю с заявлением об увольнении без отработки двух недель, ссылаясь на статью 80 ТК РФ. Однако, необходимо помнить, что работодатель может не согласиться с вашими аргументами, и в этом случае может потребоваться дополнительное обсуждение